# Data Quality & Reproducibility Audit

This notebook audits the landslide dataset pipeline before model training.
It checks:
- train/val/test balance
- missing or duplicate samples
- feature array consistency
- normalization metadata sanity
- split integrity and reproducibility

This is a contributor-grade notebook because it validates the pipeline
used by the model and makes the repository easier to trust and extend.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

PROJECT_DIR = Path("..").resolve()
PROC_DIR = PROJECT_DIR / "data" / "processed"
METRIC_DIR = PROJECT_DIR / "outputs" / "metrics"
METRIC_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_DIR)
print("Processed dir:", PROC_DIR)



In [ ]:
index = pd.read_csv(PROC_DIR / "dataset_index.csv")
X = np.load(PROC_DIR / "X.npy")
y = np.load(PROC_DIR / "y.npy")

print("dataset_index rows:", len(index))
print("X shape:", X.shape)
print("y shape:", y.shape)
print("labels positive rate:", y.mean())

In [ ]:
split_counts = index["split"].value_counts()
print("Split counts:")
print(split_counts)

for split_name in ["train", "val", "test"]:
    mask = index["split"] == split_name
    if mask.sum() > 0:
        rate = y[mask].mean()
        print(f"{split_name}: {mask.sum()} rows, positive rate = {rate:.3%}")

In [ ]:
missing_rows = index[index.isna().any(axis=1)]
duplicate_rows = index[index.duplicated(subset=["landslide_id"], keep=False)]

print("Missing rows:", len(missing_rows))
print("Duplicate landslide IDs:", len(duplicate_rows))

if len(missing_rows):
    print(missing_rows.head())

if len(duplicate_rows):
    print(duplicate_rows.head())

In [ ]:
with open(PROC_DIR / "norm_stats.json", "r") as f:
    stats = json.load(f)

print("Channels:", stats.get("channels"))
print("Train count:", stats.get("n_train"))
print("Mean length:", len(stats.get("mean", [])))
print("Std length:", len(stats.get("std", [])))

assert len(stats.get("channels", [])) == X.shape[1], "Channel count mismatch with X.npy"
assert stats.get("n_train") is not None, "Missing n_train in norm_stats.json"

In [ ]:
assert len(index) == len(y), "Dataset index and labels length mismatch"
assert X.shape[0] == len(index), "Feature array length mismatch with dataset index"
assert X.shape[0] == len(y), "Feature array and labels length mismatch"

print("All core integrity checks passed.")

In [ ]:
# Quick split leakage check: no sample should appear in more than one split
dupe_split = index.duplicated(subset=["landslide_id", "split"], keep=False).sum()
print("Samples duplicated within same split:", dupe_split)

# check if an ID appears in multiple splits
cross_split = index.groupby("landslide_id")["split"].nunique().gt(1).sum()
print("IDs appearing in more than one split:", cross_split)

In [ ]:
audit_summary = {
    "n_total": int(len(index)),
    "n_train": int((index["split"] == "train").sum()),
    "n_val": int((index["split"] == "val").sum()),
    "n_test": int((index["split"] == "test").sum()),
    "positive_rate_total": float(y.mean()),
    "positive_rate_train": float(y[index["split"] == "train"].mean()),
    "positive_rate_val": float(y[index["split"] == "val"].mean()),
    "positive_rate_test": float(y[index["split"] == "test"].mean()),
    "missing_rows": int(index.isna().any(axis=1).sum()),
    "duplicate_ids": int(index.duplicated(subset=["landslide_id"], keep=False).sum()),
    "cross_split_ids": int(index.groupby("landslide_id")["split"].nunique().gt(1).sum()),
    "channel_count": int(X.shape[1]),
}

pd.DataFrame([audit_summary]).to_csv(METRIC_DIR / "data_quality_audit.csv", index=False)
print(audit_summary)

## Findings summary

This audit checks that:
- the processed dataset is internally consistent
- the split sizes are sensible
- the class ratio is stable enough for training
- no duplicates or cross-split leakage are present
- normalization metadata matches the data shape

This notebook can be used as a reproducibility gate before training or
before opening a pull request that changes the dataset pipeline.